# Module 1.3: Object-Oriented Python
## Python Foundations for Swift Engineers

This notebook maps your deep Swift OOP and protocol-oriented knowledge to Python's object model. You will find that Python's approach is more flexible (and less safe) than Swift's, with powerful features like multiple inheritance, magic methods, and structural subtyping.

**Prerequisites**: Completed Module 1.1 and 1.2  
**Time estimate**: 75-100 minutes

## Swift vs Python Quick Reference

| Concept | Swift | Python |
|---|---|---|
| Class definition | `class Dog { }` | `class Dog:` |
| Initializer | `init(name: String)` | `def __init__(self, name: str):` |
| Self reference | `self` (implicit) | `self` (explicit, first param) |
| Stored property | `var name: String` | `self.name = name` (in `__init__`) |
| Computed property | `var desc: String { get { } }` | `@property` decorator |
| Inheritance | `class Puppy: Dog` | `class Puppy(Dog):` |
| Protocol | `protocol Drawable { }` | `class Drawable(Protocol):` |
| Abstract class | N/A (use protocol) | `class Base(ABC):` |
| Struct (value type) | `struct Point { }` | `@dataclass(frozen=True)` |
| Enum | `enum Color { case red }` | `class Color(Enum):` |
| String conversion | `description` / `CustomStringConvertible` | `__str__` / `__repr__` |
| Equality | `Equatable` / `==` | `__eq__` |
| Hashable | `Hashable` | `__hash__` |
| Subscript | `subscript(i: Int)` | `__getitem__` / `__setitem__` |
| deinit | `deinit { }` | `__del__` (rarely used) |
| Access control | `private`, `internal`, `public` | `_private`, `__mangled` (conventions) |

---
## 1. Classes: Swift class vs Python class

### Swift Mindset
```swift
class Dog {
    let name: String
    var age: Int
    
    init(name: String, age: Int) {
        self.name = name
        self.age = age
    }
    
    func bark() -> String {
        return "\(name) says Woof!"
    }
}
```

### Python Approach
- `self` is always the **explicit** first parameter (Swift makes it implicit)
- No separate property declarations -- attributes are created in `__init__`
- No access control keywords -- conventions use `_` prefix for private

In [ ]:
class Dog:
    """A simple dog class - Python equivalent of a Swift class."""
    
    # Class variable (like Swift's static property)
    species: str = "Canis familiaris"
    
    def __init__(self, name: str, age: int) -> None:
        """Initializer (Swift: init).
        
        self is explicit - it's how Python accesses instance attributes.
        No 'var name: String' declaration needed - just assign in __init__.
        """
        self.name = name      # Instance variable (public)
        self.age = age        # Instance variable (public)
        self._energy = 100    # Convention: _ prefix means "private" (but still accessible!)
    
    def bark(self) -> str:
        """Instance method. Note: self is always the first parameter."""
        self._energy -= 10
        return f"{self.name} says Woof!"
    
    def __str__(self) -> str:
        """Like Swift's CustomStringConvertible.description."""
        return f"Dog(name='{self.name}', age={self.age})"
    
    def __repr__(self) -> str:
        """Developer-facing string (used in debugger, collections)."""
        return f"Dog(name={self.name!r}, age={self.age!r})"


# Creating instances (no 'new' keyword)
rex = Dog("Rex", 5)
luna = Dog(name="Luna", age=3)  # keyword arguments work too

print(rex.bark())
print(luna)
print(f"Species: {Dog.species}")       # Access class variable
print(f"Also: {rex.species}")          # Instance can also access it
print(f"Energy: {rex._energy}")        # "private" but still accessible!
print(f"repr: {rex!r}")

In [ ]:
# Access control: conventions vs enforcement

class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner            # Public
        self._balance = balance       # Protected (convention: single underscore)
        self.__pin = 1234             # Name mangled (double underscore)
    
    def get_balance(self) -> float:
        return self._balance

acct = BankAccount("Daniel", 1000.0)

print(f"Owner: {acct.owner}")           # Fine
print(f"Balance: {acct._balance}")      # Works but linters will warn

# Double underscore causes "name mangling" - renamed to _ClassName__attr
try:
    print(acct.__pin)  # AttributeError!
except AttributeError as e:
    print(f"Error: {e}")

# But you CAN still access it (Python trusts the developer)
print(f"Mangled name: {acct._BankAccount__pin}")  # Works!

# Bottom line: Python has NO true private access. It's all conventions.
# Philosophy: "We're all consenting adults here."

---
## 2. Properties (@property vs Swift Computed Properties)

### Swift Mindset
```swift
class Circle {
    var radius: Double
    
    var area: Double {           // computed property
        return .pi * radius * radius
    }
    
    var diameter: Double {       // computed with setter
        get { return radius * 2 }
        set { radius = newValue / 2 }
    }
}
```

In [ ]:
import math

class Circle:
    def __init__(self, radius: float) -> None:
        self._radius = radius
    
    # Read-only computed property (like Swift's get-only computed property)
    @property
    def area(self) -> float:
        return math.pi * self._radius ** 2
    
    # Property with getter and setter (like Swift's get/set)
    @property
    def radius(self) -> float:
        return self._radius
    
    @radius.setter
    def radius(self, value: float) -> None:
        if value < 0:
            raise ValueError("Radius cannot be negative")
        self._radius = value
    
    @property
    def diameter(self) -> float:
        return self._radius * 2
    
    @diameter.setter
    def diameter(self, value: float) -> None:
        self._radius = value / 2


c = Circle(5.0)
print(f"Radius: {c.radius}")       # Calls the getter
print(f"Area: {c.area:.2f}")       # Calls the computed property
print(f"Diameter: {c.diameter}")   # Calls the getter

c.diameter = 20                     # Calls the setter
print(f"\nAfter setting diameter=20:")
print(f"Radius: {c.radius}")       # 10.0
print(f"Area: {c.area:.2f}")       # ~314.16

# Validation in setter
try:
    c.radius = -5
except ValueError as e:
    print(f"\nError: {e}")

# area has no setter, so it's read-only
try:
    c.area = 100
except AttributeError as e:
    print(f"Error: {e}")

---
## 3. Inheritance and MRO

### Swift Mindset
```swift
class Animal { }                       // base class
class Dog: Animal { }                   // single inheritance
// No multiple class inheritance in Swift - use protocols instead
```

### Python Differences
- Python supports **multiple inheritance** (Swift does not)
- Method Resolution Order (MRO) determines which parent's method is called
- `super()` follows the MRO, not just the immediate parent

In [ ]:
# Single inheritance (works like Swift)
class Animal:
    def __init__(self, name: str) -> None:
        self.name = name
    
    def speak(self) -> str:
        return f"{self.name} makes a sound"
    
    def __str__(self) -> str:
        return f"{type(self).__name__}(name='{self.name}')"


class Dog(Animal):   # Inherits from Animal (Swift: class Dog: Animal)
    def __init__(self, name: str, breed: str) -> None:
        super().__init__(name)   # Swift: super.init(name: name)
        self.breed = breed
    
    def speak(self) -> str:      # Override (no 'override' keyword needed)
        return f"{self.name} says Woof!"
    
    def fetch(self, item: str) -> str:
        return f"{self.name} fetches the {item}"


class GuideDog(Dog):  # Multi-level inheritance
    def __init__(self, name: str, breed: str, handler: str) -> None:
        super().__init__(name, breed)
        self.handler = handler
    
    def guide(self) -> str:
        return f"{self.name} guides {self.handler}"


buddy = GuideDog("Buddy", "Labrador", "Alice")
print(buddy.speak())    # From Dog
print(buddy.fetch("ball"))  # From Dog
print(buddy.guide())    # From GuideDog
print(buddy)            # From Animal (__str__)

# isinstance and issubclass (like Swift's 'is' and type checking)
print(f"\nisinstance(buddy, Dog): {isinstance(buddy, Dog)}")
print(f"isinstance(buddy, Animal): {isinstance(buddy, Animal)}")
print(f"issubclass(GuideDog, Animal): {issubclass(GuideDog, Animal)}")

In [ ]:
# Multiple inheritance - Python's power (and potential chaos)
# This has NO equivalent in Swift (you'd use protocols instead)

class Flyable:
    def fly(self) -> str:
        return f"{self.name} is flying"  # type: ignore

class Swimmable:
    def swim(self) -> str:
        return f"{self.name} is swimming"  # type: ignore

class Duck(Animal, Flyable, Swimmable):
    def speak(self) -> str:
        return f"{self.name} says Quack!"


donald = Duck("Donald")
print(donald.speak())   # From Duck
print(donald.fly())     # From Flyable
print(donald.swim())    # From Swimmable

# Method Resolution Order (MRO) - the order Python searches for methods
print(f"\nDuck MRO: {[cls.__name__ for cls in Duck.__mro__]}")
# Duck -> Animal -> Flyable -> Swimmable -> object

In [ ]:
# The Diamond Problem - MRO in action

class Base:
    def greet(self) -> str:
        return "Hello from Base"

class Left(Base):
    def greet(self) -> str:
        return "Hello from Left"

class Right(Base):
    def greet(self) -> str:
        return "Hello from Right"

class Child(Left, Right):  # Diamond: Child -> Left -> Right -> Base
    pass

c = Child()
print(f"Child.greet(): {c.greet()}")  # "Hello from Left" (Left listed first)
print(f"MRO: {[cls.__name__ for cls in Child.__mro__]}")

# Cooperative multiple inheritance with super()
class Base2:
    def greet(self) -> str:
        return "Base"

class Left2(Base2):
    def greet(self) -> str:
        return f"Left -> {super().greet()}"

class Right2(Base2):
    def greet(self) -> str:
        return f"Right -> {super().greet()}"

class Child2(Left2, Right2):
    def greet(self) -> str:
        return f"Child -> {super().greet()}"

print(f"\nCooperative: {Child2().greet()}")
# Child -> Left -> Right -> Base (super() follows MRO!)

---
## 4. Magic Methods (Dunder Methods)

Magic methods (double-underscore or "dunder" methods) are Python's equivalent of Swift's protocol conformances. They let you customize how objects behave with built-in operations.

### Swift Comparison
| Python Magic Method | Swift Equivalent |
|---|---|
| `__str__` | `CustomStringConvertible.description` |
| `__repr__` | `CustomDebugStringConvertible.debugDescription` |
| `__eq__` | `Equatable.==` |
| `__hash__` | `Hashable.hash(into:)` |
| `__lt__`, `__le__`, `__gt__`, `__ge__` | `Comparable.<`, `<=`, `>`, `>=` |
| `__len__` | `Collection.count` |
| `__getitem__` | `subscript { get }` |
| `__setitem__` | `subscript { set }` |
| `__contains__` | `Sequence.contains()` |
| `__iter__` | `Sequence.makeIterator()` |
| `__add__` | `static func +(lhs:rhs:)` |
| `__bool__` | N/A (Swift requires explicit Bool) |

In [ ]:
class Vector:
    """A 2D vector class demonstrating common magic methods.
    
    In Swift, you'd conform to Equatable, Hashable, CustomStringConvertible,
    and implement static func + and other operators.
    """
    
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y
    
    # String representations
    def __str__(self) -> str:
        """User-friendly string (print(), str(), f-string)."""
        return f"({self.x}, {self.y})"
    
    def __repr__(self) -> str:
        """Developer-friendly string (debugger, REPL, inside containers)."""
        return f"Vector({self.x!r}, {self.y!r})"
    
    # Equality and hashing (like Swift's Equatable + Hashable)
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Vector):
            return NotImplemented
        return self.x == other.x and self.y == other.y
    
    def __hash__(self) -> int:
        return hash((self.x, self.y))
    
    # Arithmetic operators
    def __add__(self, other: "Vector") -> "Vector":
        return Vector(self.x + other.x, self.y + other.y)
    
    def __sub__(self, other: "Vector") -> "Vector":
        return Vector(self.x - other.x, self.y - other.y)
    
    def __mul__(self, scalar: float) -> "Vector":
        return Vector(self.x * scalar, self.y * scalar)
    
    def __rmul__(self, scalar: float) -> "Vector":
        """Handles scalar * vector (when vector is on the right)."""
        return self.__mul__(scalar)
    
    # Comparison
    def __abs__(self) -> float:
        """Magnitude (like abs() for numbers)."""
        return (self.x ** 2 + self.y ** 2) ** 0.5
    
    # Boolean
    def __bool__(self) -> bool:
        """Is this a non-zero vector?"""
        return self.x != 0 or self.y != 0


# Using it
v1 = Vector(3, 4)
v2 = Vector(1, 2)

print(f"v1 = {v1}")              # __str__
print(f"v1 + v2 = {v1 + v2}")    # __add__
print(f"v1 - v2 = {v1 - v2}")    # __sub__
print(f"v1 * 3 = {v1 * 3}")      # __mul__
print(f"3 * v1 = {3 * v1}")      # __rmul__
print(f"|v1| = {abs(v1)}")       # __abs__
print(f"v1 == v2: {v1 == v2}")   # __eq__
print(f"v1 == Vector(3,4): {v1 == Vector(3, 4)}")  # __eq__

# Hashable means we can use it in sets and as dict keys
vectors = {v1, v2, Vector(3, 4)}  # duplicate removed!
print(f"\nUnique vectors: {vectors}")

# Boolean context
zero = Vector(0, 0)
print(f"\nbool(v1): {bool(v1)}")
print(f"bool(zero): {bool(zero)}")

In [ ]:
# __len__ and __getitem__ - make your class behave like a collection
# Swift equivalent: conforming to Collection protocol with subscript

class Playlist:
    def __init__(self, name: str, songs: list[str] | None = None) -> None:
        self.name = name
        self._songs = songs or []
    
    def __len__(self) -> int:
        """len(playlist) - like Swift's count property."""
        return len(self._songs)
    
    def __getitem__(self, index: int) -> str:
        """playlist[i] - like Swift's subscript."""
        return self._songs[index]
    
    def __setitem__(self, index: int, value: str) -> None:
        """playlist[i] = 'song' - like Swift's subscript setter."""
        self._songs[index] = value
    
    def __contains__(self, song: str) -> bool:
        """'song' in playlist - like Swift's contains()."""
        return song in self._songs
    
    def __iter__(self):
        """for song in playlist: - makes it iterable."""
        return iter(self._songs)
    
    def __repr__(self) -> str:
        return f"Playlist({self.name!r}, {len(self)} songs)"


playlist = Playlist("Road Trip", ["Bohemian Rhapsody", "Hotel California", "Stairway to Heaven"])

print(f"Playlist: {playlist}")
print(f"Length: {len(playlist)}")
print(f"First song: {playlist[0]}")
print(f"Last song: {playlist[-1]}")
print(f"Contains 'Hotel California': {'Hotel California' in playlist}")

print("\nAll songs:")
for song in playlist:  # Uses __iter__
    print(f"  - {song}")

# Slicing also works if __getitem__ accepts slices
playlist[0] = "Imagine"  # Uses __setitem__
print(f"\nAfter edit: {list(playlist)}")

---
## 5. Abstract Base Classes (ABCs) vs Swift Protocols

### Swift Mindset
```swift
protocol Shape {
    var area: Double { get }
    func describe() -> String
}

class Circle: Shape {
    var area: Double { .pi * radius * radius }
    func describe() -> String { "Circle with radius \(radius)" }
}
```

### Python ABCs
ABCs are Python's closest equivalent to Swift protocols with **required methods**. They enforce that subclasses implement certain methods at instantiation time.

In [ ]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    """Abstract base class - like a Swift protocol.
    
    You CANNOT instantiate this directly.
    Subclasses MUST implement all @abstractmethod methods.
    """
    
    @property
    @abstractmethod
    def area(self) -> float:
        """Required computed property."""
        ...
    
    @property
    @abstractmethod
    def perimeter(self) -> float:
        """Required computed property."""
        ...
    
    @abstractmethod
    def describe(self) -> str:
        """Required method."""
        ...
    
    # Non-abstract methods provide default implementations
    # (like Swift protocol extensions)
    def is_larger_than(self, other: "Shape") -> bool:
        return self.area > other.area


class CircleShape(Shape):
    def __init__(self, radius: float) -> None:
        self.radius = radius
    
    @property
    def area(self) -> float:
        return math.pi * self.radius ** 2
    
    @property
    def perimeter(self) -> float:
        return 2 * math.pi * self.radius
    
    def describe(self) -> str:
        return f"Circle with radius {self.radius}"


class Rectangle(Shape):
    def __init__(self, width: float, height: float) -> None:
        self.width = width
        self.height = height
    
    @property
    def area(self) -> float:
        return self.width * self.height
    
    @property
    def perimeter(self) -> float:
        return 2 * (self.width + self.height)
    
    def describe(self) -> str:
        return f"Rectangle {self.width}x{self.height}"


# Can't instantiate abstract class
try:
    s = Shape()
except TypeError as e:
    print(f"Can't instantiate ABC: {e}")

# Using concrete classes
circle = CircleShape(5)
rect = Rectangle(4, 6)

print(f"\n{circle.describe()}: area={circle.area:.2f}, perimeter={circle.perimeter:.2f}")
print(f"{rect.describe()}: area={rect.area:.2f}, perimeter={rect.perimeter:.2f}")
print(f"Circle larger than rectangle? {circle.is_larger_than(rect)}")

# Polymorphism (like Swift's protocol-typed arrays)
shapes: list[Shape] = [CircleShape(3), Rectangle(4, 5), CircleShape(7)]
for shape in shapes:
    print(f"  {shape.describe()}: area = {shape.area:.2f}")

---
## 6. Dataclasses vs Swift Structs

### Swift Mindset
```swift
struct Point: Equatable, Hashable {
    let x: Double
    let y: Double
    // init, ==, hash(into:) are auto-synthesized!
}
```

### Python Dataclasses
Dataclasses auto-generate `__init__`, `__repr__`, `__eq__`, and optionally `__hash__`. With `frozen=True`, they approximate Swift struct value semantics.

In [ ]:
from dataclasses import dataclass, field

# Basic dataclass (like a Swift struct with auto-synthesized members)
@dataclass
class Point:
    x: float
    y: float
    
    # Auto-generated: __init__, __repr__, __eq__
    # You can still add methods:
    def distance_to(self, other: "Point") -> float:
        return ((self.x - other.x) ** 2 + (self.y - other.y) ** 2) ** 0.5


p1 = Point(3.0, 4.0)
p2 = Point(0.0, 0.0)

print(f"p1 = {p1}")                              # Auto __repr__
print(f"p1 == Point(3.0, 4.0): {p1 == Point(3.0, 4.0)}")  # Auto __eq__
print(f"Distance: {p1.distance_to(p2)}")         # Custom method

# Mutable by default (unlike Swift struct with let)
p1.x = 10.0
print(f"After mutation: {p1}")

In [ ]:
# Frozen dataclass - immutable like a Swift struct with all `let` properties
@dataclass(frozen=True)
class FrozenPoint:
    x: float
    y: float
    # frozen=True also generates __hash__, so it can be used in sets/dicts

fp = FrozenPoint(3.0, 4.0)
print(f"Frozen: {fp}")
print(f"Hash: {hash(fp)}")

try:
    fp.x = 10.0  # Cannot mutate!
except AttributeError as e:
    print(f"Error: {e}")

# Can be used in sets (because it's hashable)
points = {FrozenPoint(1, 2), FrozenPoint(3, 4), FrozenPoint(1, 2)}
print(f"Unique points: {points}")  # Duplicate removed

In [ ]:
# Advanced dataclass features

@dataclass
class Player:
    name: str
    score: int = 0                           # Default value
    inventory: list[str] = field(default_factory=list)  # Mutable default
    _id: int = field(init=False, repr=False)  # Excluded from init and repr
    
    _next_id: int = field(init=False, repr=False, default=0)
    
    def __post_init__(self) -> None:
        """Called after __init__ - like Swift's didSet or post-init setup."""
        Player._next_id += 1
        self._id = Player._next_id


p1 = Player("Alice")
p2 = Player("Bob", score=100, inventory=["sword", "shield"])

print(f"p1: {p1}")
print(f"p2: {p2}")

# IMPORTANT: Why field(default_factory=list)?
# If you wrote `inventory: list[str] = []`, ALL instances would share the SAME list!
# default_factory creates a NEW list for each instance (like Swift's default values)

In [ ]:
# Dataclass with ordering (like Swift's Comparable)
@dataclass(order=True)
class Student:
    gpa: float           # Compared first (fields are compared in order)
    name: str = field(compare=False)  # Excluded from comparisons
    student_id: int = field(compare=False)

students = [
    Student(3.8, "Alice", 101),
    Student(3.9, "Bob", 102),
    Student(3.5, "Charlie", 103),
    Student(3.9, "Diana", 104),
]

# Sorting works automatically
for s in sorted(students, reverse=True):
    print(f"  {s.name}: GPA {s.gpa}")

# Comparison operators work
print(f"\nAlice < Bob: {students[0] < students[1]}")

---
## 7. Enums

### Swift Mindset
```swift
enum Direction: String {
    case north, south, east, west
}

enum Planet: Int {
    case mercury = 1, venus, earth
}

// Swift enums with associated values:
enum Result<T> {
    case success(T)
    case failure(Error)
}
```

### Python Enums
Python enums are simpler than Swift's (no associated values), but cover most use cases.

In [ ]:
from enum import Enum, IntEnum, auto, Flag

# Basic enum (like Swift enum with raw values)
class Direction(Enum):
    NORTH = "north"
    SOUTH = "south"
    EAST = "east"
    WEST = "west"

# Usage
d = Direction.NORTH
print(f"Direction: {d}")         # Direction.NORTH
print(f"Name: {d.name}")        # NORTH
print(f"Value: {d.value}")      # north

# Create from value (like Swift's init(rawValue:))
d2 = Direction("south")
print(f"From value: {d2}")

# Comparison
print(f"d == Direction.NORTH: {d == Direction.NORTH}")
print(f"d is Direction.NORTH: {d is Direction.NORTH}")

# Iteration (like Swift's CaseIterable)
print(f"All: {[d.value for d in Direction]}")

In [ ]:
# IntEnum - for integer values
class Priority(IntEnum):
    LOW = auto()      # auto() assigns 1, 2, 3, ... (like Swift's implicit raw values)
    MEDIUM = auto()
    HIGH = auto()
    CRITICAL = auto()

print(f"Priority.HIGH = {Priority.HIGH}")
print(f"Value: {Priority.HIGH.value}")

# IntEnum can be compared with ints (unlike regular Enum)
print(f"HIGH > MEDIUM: {Priority.HIGH > Priority.MEDIUM}")
print(f"HIGH == 3: {Priority.HIGH == 3}")

In [ ]:
# Enum with methods (like Swift enum with functions)
class Color(Enum):
    RED = (255, 0, 0)
    GREEN = (0, 255, 0)
    BLUE = (0, 0, 255)
    WHITE = (255, 255, 255)
    BLACK = (0, 0, 0)
    
    @property
    def hex_code(self) -> str:
        r, g, b = self.value
        return f"#{r:02x}{g:02x}{b:02x}"
    
    @property
    def is_bright(self) -> bool:
        r, g, b = self.value
        return (r + g + b) / 3 > 128

for color in Color:
    print(f"{color.name:6s}: {color.hex_code} {'(bright)' if color.is_bright else '(dark)'}")

# Pattern matching with enums
def describe_color(c: Color) -> str:
    match c:
        case Color.RED:
            return "Warm color"
        case Color.BLUE:
            return "Cool color"
        case _:
            return "Other color"

print(f"\n{describe_color(Color.RED)}")

---
## 8. Protocols via typing.Protocol (Structural Subtyping)

### Swift Mindset
```swift
protocol Drawable {
    func draw() -> String
}
// Any class/struct that implements draw() can conform
// But conformance must be DECLARED: class Circle: Drawable
```

### Python's typing.Protocol
Python 3.8+ supports **structural subtyping** (duck typing with type checking). If a class has the right methods, it satisfies the Protocol **without explicitly inheriting from it**. This is closer to Go's interfaces or Swift's planned `~Copyable`-style implicit conformance.

In [ ]:
from typing import Protocol, runtime_checkable

# Define a Protocol (like a Swift protocol, but structural)
@runtime_checkable  # Allows isinstance() checks
class Drawable(Protocol):
    def draw(self) -> str:
        ...  # Ellipsis means "abstract" (no implementation)

@runtime_checkable
class Resizable(Protocol):
    def resize(self, factor: float) -> None:
        ...

# These classes DON'T inherit from Drawable/Resizable!
# They satisfy the protocol just by having the right methods.

class CircleWidget:
    def __init__(self, radius: float) -> None:
        self.radius = radius
    
    def draw(self) -> str:  # Satisfies Drawable
        return f"Drawing circle with radius {self.radius}"
    
    def resize(self, factor: float) -> None:  # Satisfies Resizable
        self.radius *= factor


class TextLabel:
    def __init__(self, text: str) -> None:
        self.text = text
    
    def draw(self) -> str:  # Satisfies Drawable
        return f"Drawing text: '{self.text}'"
    # No resize - does NOT satisfy Resizable


# Using Protocol as type hint
def render(item: Drawable) -> None:
    print(item.draw())

# Both work even though they don't explicitly inherit from Drawable
render(CircleWidget(5.0))
render(TextLabel("Hello"))

# runtime_checkable allows isinstance
circle = CircleWidget(5.0)
label = TextLabel("Hi")

print(f"\nCircle is Drawable: {isinstance(circle, Drawable)}")
print(f"Label is Drawable: {isinstance(label, Drawable)}")
print(f"Circle is Resizable: {isinstance(circle, Resizable)}")
print(f"Label is Resizable: {isinstance(label, Resizable)}")

In [ ]:
# Combining ABCs (for enforcement) with Protocols (for flexibility)
# This is the Python equivalent of Swift's protocol + protocol extension pattern

from typing import Protocol
from abc import ABC, abstractmethod

# When you WANT to enforce implementation, use ABC:
class Repository(ABC):
    """ABC for when you need guaranteed implementation."""
    @abstractmethod
    def save(self, data: dict) -> None: ...
    
    @abstractmethod
    def load(self, key: str) -> dict: ...
    
    # Default implementation (like Swift protocol extension)
    def exists(self, key: str) -> bool:
        try:
            self.load(key)
            return True
        except KeyError:
            return False

# When you want duck typing compatibility, use Protocol:
class Serializable(Protocol):
    """Protocol for structural subtyping - no inheritance needed."""
    def to_dict(self) -> dict: ...
    def from_dict(self, data: dict) -> "Serializable": ...

# Concrete implementation of the ABC
class InMemoryRepository(Repository):
    def __init__(self) -> None:
        self._store: dict[str, dict] = {}
    
    def save(self, data: dict) -> None:
        key = data.get("id", str(id(data)))
        self._store[key] = data
    
    def load(self, key: str) -> dict:
        if key not in self._store:
            raise KeyError(f"Key '{key}' not found")
        return self._store[key]


repo = InMemoryRepository()
repo.save({"id": "user1", "name": "Daniel"})
print(f"Loaded: {repo.load('user1')}")
print(f"Exists: {repo.exists('user1')}")
print(f"Missing: {repo.exists('user2')}")

---
## Exercises

These exercises bring together all the OOP concepts covered. Each one has a practical application.

### Exercise 1: Card and Deck Classes

Design a `Card` and `Deck` class:
- `Card` should be a frozen dataclass with `suit` and `rank`
- `Card` should have a nice string representation (e.g., "A of Spades")
- `Deck` should support `len()`, iteration, indexing, and `shuffle()`/`deal(n)`
- Use an Enum for suits

Think about this like designing a Swift struct for Card and a class for Deck.

In [ ]:
# YOUR CODE HERE
# Implement Card (frozen dataclass) and Deck classes
# Card should support: str(), repr(), ==, hash, comparison by rank
# Deck should support: len(), iteration, indexing, shuffle(), deal(n)

In [ ]:
# SOLUTION - Exercise 1

from dataclasses import dataclass
from enum import Enum
import random

class Suit(Enum):
    HEARTS = "Hearts"
    DIAMONDS = "Diamonds"
    CLUBS = "Clubs"
    SPADES = "Spades"

RANK_NAMES = {
    1: "A", 2: "2", 3: "3", 4: "4", 5: "5", 6: "6", 7: "7",
    8: "8", 9: "9", 10: "10", 11: "J", 12: "Q", 13: "K"
}

@dataclass(frozen=True, order=True)
class Card:
    rank: int                             # 1-13 (A=1, J=11, Q=12, K=13)
    suit: Suit = dataclass_field = None   # Comparison ignores suit by default
    
    def __init__(self, rank: int, suit: Suit):
        # frozen dataclass requires object.__setattr__ for init
        object.__setattr__(self, 'rank', rank)
        object.__setattr__(self, 'suit', suit)
    
    def __str__(self) -> str:
        return f"{RANK_NAMES[self.rank]} of {self.suit.value}"
    
    def __repr__(self) -> str:
        return f"Card({RANK_NAMES[self.rank]}, {self.suit.value})"


class Deck:
    def __init__(self) -> None:
        self._cards = [
            Card(rank, suit)
            for suit in Suit
            for rank in range(1, 14)
        ]
    
    def __len__(self) -> int:
        return len(self._cards)
    
    def __getitem__(self, index: int) -> Card:
        return self._cards[index]
    
    def __iter__(self):
        return iter(self._cards)
    
    def __repr__(self) -> str:
        return f"Deck({len(self)} cards)"
    
    def shuffle(self) -> None:
        random.shuffle(self._cards)
    
    def deal(self, n: int = 1) -> list[Card]:
        if n > len(self._cards):
            raise ValueError(f"Cannot deal {n} cards, only {len(self._cards)} remaining")
        dealt = self._cards[:n]
        self._cards = self._cards[n:]
        return dealt


# Test it
deck = Deck()
print(f"New deck: {deck}")
print(f"First card: {deck[0]}")
print(f"Last card: {deck[-1]}")

deck.shuffle()
hand = deck.deal(5)
print(f"\nDealt hand: {hand}")
print(f"Remaining: {deck}")

# Cards are hashable (frozen) so they can go in sets
hand_set = set(hand)
print(f"Unique cards in hand: {len(hand_set)}")

### Exercise 2: Stack with Protocol

Implement a `Stack` class that conforms to a `Stackable` Protocol:

```python
class Stackable(Protocol):
    def push(self, item) -> None: ...
    def pop(self): ...  # Returns item or raises
    def peek(self): ... # Returns top without removing
    def __len__(self) -> int: ...
    def __bool__(self) -> bool: ...
```

Also implement a `MinStack` that tracks the minimum element in O(1) time.

In [ ]:
# YOUR CODE HERE
# Implement Stack and MinStack
# Stack: push, pop, peek, len, bool, repr, iteration
# MinStack: all of the above + get_min() in O(1)

In [ ]:
# SOLUTION - Exercise 2

from typing import Protocol, TypeVar, Generic, Iterator

T = TypeVar('T')

class Stackable(Protocol[T]):
    def push(self, item: T) -> None: ...
    def pop(self) -> T: ...
    def peek(self) -> T: ...
    def __len__(self) -> int: ...
    def __bool__(self) -> bool: ...


class Stack(Generic[T]):
    """Generic stack - satisfies Stackable protocol structurally."""
    
    def __init__(self) -> None:
        self._items: list[T] = []
    
    def push(self, item: T) -> None:
        self._items.append(item)
    
    def pop(self) -> T:
        if not self._items:
            raise IndexError("Pop from empty stack")
        return self._items.pop()
    
    def peek(self) -> T:
        if not self._items:
            raise IndexError("Peek at empty stack")
        return self._items[-1]
    
    def __len__(self) -> int:
        return len(self._items)
    
    def __bool__(self) -> bool:
        return bool(self._items)
    
    def __repr__(self) -> str:
        return f"Stack({self._items})"
    
    def __iter__(self) -> Iterator[T]:
        return reversed(self._items)  # Top of stack first


class MinStack:
    """Stack that also tracks minimum element in O(1).
    Uses a parallel stack of minimums.
    """
    
    def __init__(self) -> None:
        self._items: list[int] = []
        self._mins: list[int] = []  # Parallel stack tracking min at each level
    
    def push(self, item: int) -> None:
        self._items.append(item)
        current_min = min(item, self._mins[-1]) if self._mins else item
        self._mins.append(current_min)
    
    def pop(self) -> int:
        if not self._items:
            raise IndexError("Pop from empty stack")
        self._mins.pop()
        return self._items.pop()
    
    def peek(self) -> int:
        if not self._items:
            raise IndexError("Peek at empty stack")
        return self._items[-1]
    
    def get_min(self) -> int:
        if not self._mins:
            raise IndexError("Min of empty stack")
        return self._mins[-1]
    
    def __len__(self) -> int:
        return len(self._items)
    
    def __bool__(self) -> bool:
        return bool(self._items)
    
    def __repr__(self) -> str:
        return f"MinStack({self._items})"


# Test Stack
stack: Stack[str] = Stack()
stack.push("a")
stack.push("b")
stack.push("c")
print(f"Stack: {stack}")
print(f"Peek: {stack.peek()}")
print(f"Pop: {stack.pop()}")
print(f"Len: {len(stack)}")
print(f"Iterate: {list(stack)}")

# Test MinStack
print("\n--- MinStack ---")
ms = MinStack()
for val in [5, 3, 7, 1, 4]:
    ms.push(val)
    print(f"  Push {val}: min = {ms.get_min()}")

for _ in range(3):
    val = ms.pop()
    print(f"  Pop {val}: min = {ms.get_min()}")

### Exercise 3: Shape Hierarchy with ABCs

Build a complete shape hierarchy:
1. Abstract `Shape` with `area`, `perimeter`, and `describe()` methods
2. `Circle`, `Rectangle`, `Triangle` concrete classes
3. `Square` as a special case of `Rectangle`
4. A `ShapeCollection` class that holds shapes and can calculate total area, find the largest, and filter by type

In [ ]:
# YOUR CODE HERE
# Build the shape hierarchy and ShapeCollection

In [ ]:
# SOLUTION - Exercise 3

from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @property
    @abstractmethod
    def area(self) -> float: ...
    
    @property
    @abstractmethod
    def perimeter(self) -> float: ...
    
    def describe(self) -> str:
        return f"{type(self).__name__}: area={self.area:.2f}, perimeter={self.perimeter:.2f}"
    
    def __repr__(self) -> str:
        return self.describe()


class Circle(Shape):
    def __init__(self, radius: float) -> None:
        self.radius = radius
    
    @property
    def area(self) -> float:
        return math.pi * self.radius ** 2
    
    @property
    def perimeter(self) -> float:
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, width: float, height: float) -> None:
        self.width = width
        self.height = height
    
    @property
    def area(self) -> float:
        return self.width * self.height
    
    @property
    def perimeter(self) -> float:
        return 2 * (self.width + self.height)


class Square(Rectangle):
    def __init__(self, side: float) -> None:
        super().__init__(side, side)
    
    def describe(self) -> str:
        return f"Square(side={self.width}): area={self.area:.2f}, perimeter={self.perimeter:.2f}"


class Triangle(Shape):
    def __init__(self, a: float, b: float, c: float) -> None:
        if a + b <= c or b + c <= a or a + c <= b:
            raise ValueError("Invalid triangle sides")
        self.a, self.b, self.c = a, b, c
    
    @property
    def area(self) -> float:
        s = self.perimeter / 2  # semi-perimeter
        return math.sqrt(s * (s - self.a) * (s - self.b) * (s - self.c))
    
    @property
    def perimeter(self) -> float:
        return self.a + self.b + self.c


class ShapeCollection:
    def __init__(self, shapes: list[Shape] | None = None) -> None:
        self._shapes = shapes or []
    
    def add(self, shape: Shape) -> None:
        self._shapes.append(shape)
    
    @property
    def total_area(self) -> float:
        return sum(s.area for s in self._shapes)
    
    def largest(self) -> Shape:
        return max(self._shapes, key=lambda s: s.area)
    
    def filter_by_type(self, shape_type: type) -> list[Shape]:
        return [s for s in self._shapes if isinstance(s, shape_type)]
    
    def __len__(self) -> int:
        return len(self._shapes)
    
    def __iter__(self):
        return iter(self._shapes)


# Test
collection = ShapeCollection()
collection.add(Circle(5))
collection.add(Rectangle(4, 6))
collection.add(Square(3))
collection.add(Triangle(3, 4, 5))
collection.add(Circle(2))

print("All shapes:")
for shape in collection:
    print(f"  {shape.describe()}")

print(f"\nTotal area: {collection.total_area:.2f}")
print(f"Largest: {collection.largest().describe()}")
print(f"Circles: {collection.filter_by_type(Circle)}")
print(f"Rectangles (including squares): {collection.filter_by_type(Rectangle)}")

### Exercise 4: LinkedList with Magic Methods

Implement a `LinkedList` class that behaves like a Python sequence:
- Support `len()`, `getitem` (indexing), `contains`, iteration
- Support `+` for concatenation
- Support `==` for comparison
- Nice `__repr__` like `LinkedList(1 -> 2 -> 3)`

In [ ]:
# YOUR CODE HERE
# Implement Node and LinkedList with full magic method support

In [ ]:
# SOLUTION - Exercise 4

from __future__ import annotations
from typing import Iterator, Any

class Node:
    __slots__ = ('value', 'next')  # Memory optimization (like Swift's no extra properties)
    
    def __init__(self, value: Any, next_node: Node | None = None) -> None:
        self.value = value
        self.next = next_node
    
    def __repr__(self) -> str:
        return f"Node({self.value!r})"


class LinkedList:
    def __init__(self, items: list | None = None) -> None:
        self._head: Node | None = None
        self._length = 0
        
        if items:
            for item in items:
                self.append(item)
    
    def append(self, value: Any) -> None:
        new_node = Node(value)
        if not self._head:
            self._head = new_node
        else:
            current = self._head
            while current.next:
                current = current.next
            current.next = new_node
        self._length += 1
    
    def prepend(self, value: Any) -> None:
        self._head = Node(value, self._head)
        self._length += 1
    
    def __len__(self) -> int:
        return self._length
    
    def __getitem__(self, index: int) -> Any:
        if index < 0:
            index += self._length
        if index < 0 or index >= self._length:
            raise IndexError("LinkedList index out of range")
        
        current = self._head
        for _ in range(index):
            current = current.next
        return current.value
    
    def __contains__(self, value: Any) -> bool:
        for item in self:
            if item == value:
                return True
        return False
    
    def __iter__(self) -> Iterator:
        current = self._head
        while current:
            yield current.value
            current = current.next
    
    def __add__(self, other: LinkedList) -> LinkedList:
        result = LinkedList(list(self))
        for item in other:
            result.append(item)
        return result
    
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, LinkedList):
            return NotImplemented
        if len(self) != len(other):
            return False
        return all(a == b for a, b in zip(self, other))
    
    def __bool__(self) -> bool:
        return self._head is not None
    
    def __repr__(self) -> str:
        values = " -> ".join(str(v) for v in self)
        return f"LinkedList({values})" if values else "LinkedList(empty)"


# Test
ll1 = LinkedList([1, 2, 3])
ll2 = LinkedList([4, 5, 6])

print(f"ll1 = {ll1}")
print(f"ll2 = {ll2}")
print(f"len(ll1) = {len(ll1)}")
print(f"ll1[0] = {ll1[0]}")
print(f"ll1[-1] = {ll1[-1]}")
print(f"2 in ll1 = {2 in ll1}")
print(f"7 in ll1 = {7 in ll1}")
print(f"ll1 + ll2 = {ll1 + ll2}")
print(f"ll1 == LinkedList([1,2,3]): {ll1 == LinkedList([1, 2, 3])}")
print(f"ll1 == ll2: {ll1 == ll2}")

ll1.prepend(0)
print(f"After prepend(0): {ll1}")

print(f"\nIteration:")
for val in ll1:
    print(f"  {val}")

### Exercise 5: Plugin System with Protocols and Enums

Design a plugin system for a text processor:
1. Define a `TextPlugin` Protocol with `name` property and `process(text: str) -> str`
2. Create at least 3 plugins: `UpperCasePlugin`, `ReversePlugin`, `CensorPlugin`
3. Create a `PluginPriority` enum (HIGH, MEDIUM, LOW)
4. Build a `TextProcessor` class that manages plugins with priorities and applies them in order

This mirrors a real-world pattern you might use in iOS development with middleware or request interceptors.

In [ ]:
# YOUR CODE HERE
# Implement the plugin system

In [ ]:
# SOLUTION - Exercise 5

from typing import Protocol, runtime_checkable
from enum import IntEnum
from dataclasses import dataclass, field

class PluginPriority(IntEnum):
    HIGH = 1
    MEDIUM = 2
    LOW = 3


@runtime_checkable
class TextPlugin(Protocol):
    @property
    def name(self) -> str: ...
    
    def process(self, text: str) -> str: ...


# Plugins don't inherit from TextPlugin - they satisfy it structurally

class UpperCasePlugin:
    @property
    def name(self) -> str:
        return "UpperCase"
    
    def process(self, text: str) -> str:
        return text.upper()


class ReversePlugin:
    @property
    def name(self) -> str:
        return "Reverse"
    
    def process(self, text: str) -> str:
        return " ".join(word[::-1] for word in text.split())


class CensorPlugin:
    def __init__(self, bad_words: list[str]) -> None:
        self._bad_words = {w.lower() for w in bad_words}
    
    @property
    def name(self) -> str:
        return "Censor"
    
    def process(self, text: str) -> str:
        words = text.split()
        return " ".join(
            "*" * len(w) if w.lower() in self._bad_words else w
            for w in words
        )


class TitleCasePlugin:
    @property
    def name(self) -> str:
        return "TitleCase"
    
    def process(self, text: str) -> str:
        return text.title()


@dataclass(order=True)
class RegisteredPlugin:
    priority: PluginPriority
    plugin: TextPlugin = field(compare=False)


class TextProcessor:
    def __init__(self) -> None:
        self._plugins: list[RegisteredPlugin] = []
    
    def register(self, plugin: TextPlugin, priority: PluginPriority = PluginPriority.MEDIUM) -> None:
        if not isinstance(plugin, TextPlugin):
            raise TypeError(f"{type(plugin).__name__} does not satisfy TextPlugin protocol")
        self._plugins.append(RegisteredPlugin(priority, plugin))
        self._plugins.sort()  # Sort by priority
    
    def unregister(self, name: str) -> None:
        self._plugins = [rp for rp in self._plugins if rp.plugin.name != name]
    
    def process(self, text: str) -> str:
        result = text
        for rp in self._plugins:
            result = rp.plugin.process(result)
            print(f"  [{rp.plugin.name} ({rp.priority.name})]: {result!r}")
        return result
    
    @property
    def plugin_names(self) -> list[str]:
        return [f"{rp.plugin.name} ({rp.priority.name})" for rp in self._plugins]


# Test
processor = TextProcessor()
processor.register(CensorPlugin(["bad", "ugly"]), PluginPriority.HIGH)
processor.register(TitleCasePlugin(), PluginPriority.LOW)
processor.register(ReversePlugin(), PluginPriority.MEDIUM)

print(f"Registered plugins: {processor.plugin_names}")
print("\nProcessing:")
result = processor.process("the bad fox did ugly things")
print(f"\nFinal result: {result!r}")

# Verify protocol checking
print(f"\nUpperCasePlugin satisfies TextPlugin: {isinstance(UpperCasePlugin(), TextPlugin)}")

---
## Key Takeaways

1. **`self` is explicit**: Every instance method takes `self` as its first parameter. There's no implicit `self` like Swift.

2. **No access control**: Python uses conventions (`_private`, `__mangled`) instead of keywords (`private`, `internal`, `public`). The philosophy is "we're all consenting adults."

3. **Magic methods are protocols**: `__eq__`, `__hash__`, `__len__`, `__getitem__`, etc. are Python's way of implementing what Swift does with protocol conformance (Equatable, Hashable, Collection).

4. **Multiple inheritance exists**: Use it carefully. Prefer composition. When you do use it, understand the MRO.

5. **ABCs vs Protocols**: ABCs enforce implementation (like Swift protocols). `typing.Protocol` enables structural/duck typing (if it has the right methods, it qualifies). Use ABCs for internal hierarchies, Protocols for public APIs.

6. **Dataclasses are your friend**: Use `@dataclass` for data-holding classes. Use `frozen=True` when you want value semantics (like Swift structs). Use `order=True` for automatic comparison operators.

7. **Enums are simpler**: Python enums don't support associated values like Swift enums. For complex cases, use dataclasses or regular classes.

**Next up**: Module 1.4 - Functions, Closures, and Decorators